In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/consolidated_pipeline/1_setup/utilities


In [0]:
print(bronze_schema,silver_schema,gold_schema)

**Broze Layer Processing**

In [0]:
dbutils.widgets.text("catalog", "fmcg" )
dbutils.widgets.text("data_source","customers")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path= f"s3://sportsbar-dp-sh/{data_source}/*.csv"

print(base_path)


In [0]:
df=spark.read\
    .option("header",True)\
    .option("inferschema",True)\
    .csv(base_path)
df=(df.withColumn("ingest_time",F.current_timestamp())
    .withColumn("file_path",F.col("_metadata.file_path"))
    .withColumn("file_name",F.col("_metadata.file_name"))
    .withColumn("file_size",F.col("_metadata.file_size"))
)
df.write.format("delta")\
    .option("delta.enableChangeDataFeed",True)\
    .option("mergeschema",True)\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

 **Silver Layer Processing**


In [0]:
df_silver=spark.sql(f""" select * from {catalog}.{bronze_schema}.{data_source} """)

print("Rows Before Duplicates dropped:", df_silver.count())
df_silver=df_silver.dropDuplicates(["customer_id"])
print("Rows After Duplicates dropped :", df_silver.count())
df_silver=df_silver.withColumn("customer_name",F.trim(F.col("customer_name")))
df_silver=df_silver.withColumn("customer_name",F.initcap(F.col("customer_name")))

# To the distinct items in the city column, create a new dataframe 
#df2=df_silver.select(F.col("city")).distinct()
#df2.show()

city_name={
    "Hyderbad" :"Hyderabad",
    "Hyderabadd" : "Hyderabad",
    "NewDelhee" : "New Delhi",
    "Bengaluruu" : "Bengaluru",
    "Bengalore" : "Bengaluru",
    "NewDelhi" : "New Delhi",
    "NewDheli" : "New Delhi"
    }
allowed=["Hyderabad","Bengaluru","New Delhi"]

df_silver = (
    df_silver
.replace(city_name, subset=["city"])\
.withColumn("city",F.when(F.col("city").isNull(),None)
            .when(F.col("city").isin(allowed),F.col("city")).otherwise(None)
))

In [0]:
customer_city={
    # sprintx Nutrition
    789403:"New Delhi",

    #Zenathlete Foods
    789420:"Bengaluru",

    #Primefuel Nutrition
    789521:"Hyderabad",
    # Recovery Lyane
    789603: "Hyderabad",

    #Primefuel Nutrition
    789522: "Hyderabad"
}

df_fix=spark.createDataFrame(
    [(k,v) for k,v in customer_city.items()],["customer_id","city"]
)

#df_fix1=df_fix.select([F.col(c).alias(f"fix_{c}") for c in df_fix.columns])

df_dp = df_fix.select([F.col(c).alias(f'dp_{c}') for c in df_fix.columns])

df_silver=(df_silver.join(df_dp, df_dp.dp_customer_id == df_silver.customer_id ,"left")\
    .withColumn(
        "city",F.coalesce(F.col("city"),F.col("dp_city"))).drop("dp_city","dp_customer_id")
    )

df_silver=df_silver.withColumn("customer_id",F.col("customer_id").cast("string"))

In [0]:
df_silver = (
    df_silver.withColumn("customer", F.concat(F.col("customer_name"), F.lit("-"), F.col("city")))
)
df_silver=(df_silver.withColumn("market",F.lit("india"))
           .withColumn("platform",F.lit("sports bar"))
           .withColumn("channel",F.lit("Aquasition"))
)
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("header",True)\
    .option("delta.enableChangeDataFeed",True)\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

**Gold Layer Processing**

In [0]:
df_gold=spark.sql(f"""select cast(customer_id as string) as customer_code,customer_name,city, customer, market, platform, channel from {catalog}.{silver_schema}.{data_source}""")
df_gold.write\
    .format("delta")\
    .mode("overwrite")\
    .option("mergeSchema",True)\
    .option("header",True)\
    .option("delta.enableChangeDataFeed",True)\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")


In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customer = spark.table("fmcg.gold.sb_dim_customers")

delta_table.alias("target").merge(
    source=df_child_customer.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()